In [25]:
import json
import os
import subprocess
import itertools
from datetime import datetime
import pandas as pd
from pathlib import Path
import sys
import pickle as pl
import torch
import itertools

In [26]:
env = os.environ.copy()
env['MKL_SERVICE_FORCE_INTEL'] = '1'
env['MKL_THREADING_LAYER'] = 'GNU'
env['OMP_NUM_THREADS'] = '1'

In [27]:
# parameter_grid_1 = {
#     "n_total": [5000],
#     "n_finetune": [2500],
#     "model_name": ["bert-base-uncased"],
#     "max_length": [256],
#     "num_labels": [6],
#     "batch_size": [32],
#     "learning_rate": [1e-2,0.003],
#     "num_epochs": [2],
#     "K": [15],
#     "lambda_min": [0.05],
#     "lambda_max": [0.95],
#     "interpolation": ["linear"],
#     "optimizer": ["Adam"],
#     "dataset": ["emotion"],
#     "proportionArr": [[0.15,0.15,0.3,0.35,0.05]]
# }

In [28]:
# computed_vectors = [[0.3, 0.25, 0.45],
# [0.3, 0.3, 0.4],
# [0.3, 0.35, 0.35] ,
# [0.3, 0.4, 0.3]  ,
# [0.3, 0.45, 0.25] ,
# [0.25, 0.3, 0.45]  ,
# [0.35, 0.3, 0.35], 
# [0.4, 0.3, 0.3] ,
# [0.45, 0.3, 0.25] ,
# [0.25, 0.45, 0.3]  ,
# [0.35, 0.35, 0.3]  ,
# [0.45, 0.25, 0.3]]

In [29]:
import re 
def parse_proportion_from_filename(filename):
    """Extract proportion array from filename like 'ag_news_6e-06_[0.1, 0.3, 0.3, 0.3].json'"""
    match = re.search(r'[\[\(]\s*([0-9eE+\-.,\s]+)\s*[\]\)]', filename)
    if match:
        arr = [float(x) for x in match.group(1).split(",") if x.strip()]
        return arr
    return None


def load_experiment_data(dataset_name,interpolation_name,base_dir):
    """
    Load all experiment results with their proportions.
    
    Args:
        base_dir: Path to directory containing experiment folders
        
    Returns:
        list: List of dicts with keys 'proportions', 'alignment_matrix', 'labels', 'output_dir'
    """
    base_path = Path(base_dir)
    
    # Find all directories that match the pattern
    all_dirs = [d for d in base_path.iterdir() if d.is_dir() and d.name.startswith(dataset_name) and interpolation_name in d.name]
    
    print(f"Scanning {len(all_dirs)} directories...")
    
    proportion_arr = []
    for output_dir in all_dirs:
        # Try to parse proportions from directory name
        proportion = parse_proportion_from_filename(output_dir.name)
        proportion_arr.append(proportion)
    return proportion_arr

In [30]:
import math

In [31]:
def generate_vectors(dataset_name,interpolation_name,base_dir,step,fixed_value,max_value,min_value,num_classes):
    
    proportion_arr = load_experiment_data(dataset_name,interpolation_name,base_dir)
    remaining_sum = 1.0 - fixed_value
    
    values = [round(step * i, 2) for i in range( math.ceil(min_value/step),   int(max_value/step) + 1)]
    print(values)
    vectors = []
    seen = set()  # Track unique combinations using sorted tuples
    num_classes = 3
    indices = list(range(num_classes))

    for fixed_idx in indices:  # Which position to fix at 0.3
        for x in values:
            y = round(remaining_sum - x, 2)
            
            # Only include if y is valid (0.05 <= y <= 0.65)
            if min_value <= y <= max_value:
                v = [0.0] * num_classes
                v[fixed_idx] = fixed_value
                
                # Fill other two positions
                other_indices = [idx for idx in indices if idx != fixed_idx]
                v[other_indices[0]] = x
                v[other_indices[1]] = y
                
                # Check if this combination (in sorted form) already exists
                v = tuple(v)
                if v not in seen and list(v) not in proportion_arr:
                    seen.add(v)
                    vectors.append(v)

    # print(f"Total unique combinations: {len(vectors)}")
    # for i, v in enumerate(vectors, 1):
    #     print(f"{i:2d}. {v}  (sum = {sum(v):.2f})")
        
    return vectors

In [32]:
step = 0.07
fixed_value = 0.3  # One class fixed at this 
max_value = 0.65
min_value = 0.05
fixed_values_arr = [0.3,0.35,0.4]

In [33]:
vectors_set = set()
for fixed_value in fixed_values_arr:
    vectors_set.update(generate_vectors('snli','linear','./results1',step,fixed_value,max_value,min_value,num_classes=3))

Scanning 25 directories...
[0.07, 0.14, 0.21, 0.28, 0.35, 0.42, 0.49, 0.56, 0.63]
Scanning 25 directories...
[0.07, 0.14, 0.21, 0.28, 0.35, 0.42, 0.49, 0.56, 0.63]
Scanning 25 directories...
[0.07, 0.14, 0.21, 0.28, 0.35, 0.42, 0.49, 0.56, 0.63]


In [34]:
vectors_list = list(vectors_set)

In [35]:
vectors_list

[(0.28, 0.3, 0.42),
 (0.14, 0.46, 0.4),
 (0.3, 0.21, 0.49),
 (0.3, 0.28, 0.42),
 (0.21, 0.35, 0.44),
 (0.56, 0.14, 0.3),
 (0.07, 0.58, 0.35),
 (0.28, 0.37, 0.35),
 (0.14, 0.35, 0.51),
 (0.21, 0.44, 0.35),
 (0.49, 0.35, 0.16),
 (0.21, 0.4, 0.39),
 (0.42, 0.18, 0.4),
 (0.07, 0.63, 0.3),
 (0.28, 0.35, 0.37),
 (0.42, 0.23, 0.35),
 (0.4, 0.21, 0.39),
 (0.4, 0.28, 0.32),
 (0.49, 0.11, 0.4),
 (0.4, 0.49, 0.11),
 (0.35, 0.21, 0.44),
 (0.35, 0.28, 0.37),
 (0.42, 0.3, 0.28),
 (0.14, 0.51, 0.35),
 (0.07, 0.3, 0.63),
 (0.14, 0.4, 0.46),
 (0.28, 0.42, 0.3),
 (0.42, 0.28, 0.3),
 (0.49, 0.16, 0.35),
 (0.56, 0.3, 0.14),
 (0.14, 0.56, 0.3),
 (0.21, 0.49, 0.3),
 (0.4, 0.35, 0.25),
 (0.28, 0.4, 0.32),
 (0.63, 0.07, 0.3),
 (0.35, 0.56, 0.09),
 (0.3, 0.42, 0.28),
 (0.07, 0.53, 0.4),
 (0.56, 0.35, 0.09),
 (0.3, 0.07, 0.63),
 (0.42, 0.35, 0.23),
 (0.07, 0.35, 0.58),
 (0.35, 0.4, 0.25),
 (0.4, 0.42, 0.18),
 (0.56, 0.09, 0.35),
 (0.21, 0.3, 0.49),
 (0.49, 0.21, 0.3),
 (0.3, 0.56, 0.14),
 (0.3, 0.49, 0.21),
 (0

In [36]:
# 0.000006 - stable learning rate

parameter_grid_2 = {
    "n_total": [5000],
    "n_finetune": [2500],
    "model_name": ["bert-base-uncased"],
    "max_length": [256],
    "num_labels": [3],
    "batch_size": [32],
    "learning_rate": [0.000006],
    "num_epochs": [2],
    "K": [15],
    "lambda_min": [0.05],
    "lambda_max": [0.95],
    "interpolations": [["linear","model_baseline"],],
    "optimizer": ["Adam"],
    "dataset": ["snli"],
    "proportionArr": vectors_list
}

# 0.000006, 0.00001

# # to be fixed
# [0.1,0.7,0.1,0.1] 

# # done 
# [0.25,0.25,0.25,0.25]
# [0.3,0.1,0.3,0.3]


    # "learning_rate": [0.000006,0.000003],



In [37]:
# parameter_grid_3 = {
#     "n_total": [5000],
#     "n_finetune": [2500],
#     "model_name": ["bert-base-uncased"],
#     "max_length": [256],
#     "num_labels": [5],
#     "batch_size": [32],
#     "learning_rate": [0.001,0.003],
#     "num_epochs": [2],
#     "K": [15],
#     "lambda_min": [0.05],
#     "lambda_max": [0.95],
#     "interpolation": ["linear"],
#     "optimizer": ["Adam"],
#     "dataset": ["yelp_review_full"],
#     "proportionArr": [[0.27,0.1,0.27,0.1,0.26],]
    
    
    # # to be fixed
    # [0.1,0.1,0.6,0.1,0.1]
    # [0.05,0.1,0.05,0.7,0.1]
    
    
    # # done 
    # [0.2,0.2,0.2,0.2,0.2]


In [38]:
parameter_grids = []
parameter_grids.append(parameter_grid_2)
# parameter_grids.append(parameter_grid_3)

In [40]:
# Or define specific combinations
specific_configs = []

In [41]:
def generate_all_combinations(param_grid):
    keys = param_grid.keys()
    values = param_grid.values()
    combinations = []
    
    for combination in itertools.product(*values):
        config = dict(zip(keys, combination))
        combinations.append(config)
    
    return combinations

def create_config_file(config, experiment_path):
    config['experiment_name'] = experiment_path
    config_path = experiment_path + ".json"
    with open(config_path, 'w') as f:
        json.dump(config, f, indent=4)
    print(f"Created config file: {config_path}")

def run_experiment(config, experiment_name):
    try:
        # Create config file
        create_config_file(config, f"{experiment_name}")
        
        print(f"\nRunning experiment: {experiment_name}")
        print(f"Config: {config}")
        
        # result = subprocess.run([sys.executable, "agnews_sample_hacking_last_layer.py", f"{experiment_name}.json"], capture_output=True, text=True,shell=True)
        result = subprocess.call([sys.executable, "agnews_sample_hacking_last_layer.py", f"{experiment_name}.json"],env=env)
        
        if result == 0:
            print(f"✅ Experiment {experiment_name} completed successfully")
        else:
            print(f"❌ Experiment {experiment_name} failed")
            # print("Error:", result.stderr)
        
        return {
            "experiment_name": experiment_name,
            "config": config,
            "success": result == 0,
            # "stdout": result.stdout,
            # "stderr": result.stderr
        }
        
    except Exception as e:
        print(f"❌ Exception in {experiment_name}: {str(e)}")
        return {
            "experiment_name": experiment_name,
            "config": config,
            "success": False,
            "error": str(e)
        }

In [42]:
configs_to_run = generate_all_combinations(parameter_grids[0])
len(configs_to_run)

65

In [ ]:
# Choose experiment mode
USE_GRID_SEARCH = True # Set to True for grid search, False for specific configs

if USE_GRID_SEARCH:
    for parameter_grid in parameter_grids:
        configs_to_run = generate_all_combinations(parameter_grid)
        print(f"Total configurations to run: {len(configs_to_run)}")
        
        # Run all experiments
        results = []
        for i, config in enumerate(configs_to_run):
            experiment_name = f"{config['dataset']}_{config['proportionArr']}"
            result = run_experiment(config, experiment_name)
            results.append(result)

        # Summary
        successful = sum(1 for r in results if r["success"])
        print(f"\n{'='*50}")
        print(f"EXPERIMENT SUMMARY")
        print(f"{'='*50}")
        print(f"Total experiments: {len(results)}")
        print(f"Successful: {successful}")
        print(f"Failed: {len(results) - successful}")
        
else:
    configs_to_run = specific_configs

Total configurations to run: 65
Created config file: snli_(0.28, 0.3, 0.42).json

Running experiment: snli_(0.28, 0.3, 0.42)
Config: {'n_total': 5000, 'n_finetune': 2500, 'model_name': 'bert-base-uncased', 'max_length': 256, 'num_labels': 3, 'batch_size': 32, 'learning_rate': 6e-06, 'num_epochs': 2, 'K': 15, 'lambda_min': 0.05, 'lambda_max': 0.95, 'interpolations': ['linear', 'model_baseline'], 'optimizer': 'Adam', 'dataset': 'snli', 'proportionArr': (0.28, 0.3, 0.42), 'experiment_name': 'snli_(0.28, 0.3, 0.42)'}


2026-02-19 00:25:14.810544: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-19 00:25:14.830963: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-02-19 00:25:14.830993: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-02-19 00:25:14.831707: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-02-19 00:25:14.835822: I tensorflow/core/platform/cpu_feature_guar

✓ Mergekit imported successfully
Loading configuration from: snli_(0.28, 0.3, 0.42).json
✓ Configuration loaded successfully

Output directory created: snli_(0.28, 0.3, 0.42)

STEP 1: Loading the snli Dataset
Full AG News training set size: 550152
Dataset features: {'premise': Value('string'), 'hypothesis': Value('string'), 'label': ClassLabel(names=['entailment', 'neutral', 'contradiction'])}
Total samples in dataset: 550152
Valid samples (label != -1): 549367
Selected subset D with 5000 samples
Label 0....samples needed 701.....needed proportion: 0.28....actual proportion: 0.2804
Label 1....samples needed 750.....needed proportion: 0.3....actual proportion: 0.3
Label 2....samples needed 1050.....needed proportion: 0.42....actual proportion: 0.42
Size of the computed finetuning set: 2501 
Fine-tuning set expected size: 2500
Size of the fixed/updated finetuning set: 2500 
Selected fine-tuning subset D' with 2500 samples

Dataset Statistics:
  Total samples |D|: 5000
  Fine-tuning sampl

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Base model loaded: bert-base-uncased
Number of parameters: 109,484,547
Trainable parameters: 109,484,547
✓ Base model saved to snli_(0.28, 0.3, 0.42)/base_model/ (for mergekit)
✓ Base state dict saved to snli_(0.28, 0.3, 0.42)/theta_base_model.pt

STEP 3: Fine-tuning on D' to Create Expert Model (θ_exp)


...

Computing sample gradients: 100%|██████████| 157/157 [12:40<00:00,  4.85s/it]


Alignment scores for λ=0.89:
  Mean: -0.076868
  Std:  0.842774
  Min:  -2.705334
  Max:  1.000000
  Mean (in D'):     0.065517
  Mean (not in D'): -0.219253
  Difference:       0.284770

----------------------------------------------------------------------
Interpolated Model 15/15 (λ=0.95)
----------------------------------------------------------------------


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Last layer direction norm ||θ_k_last - θ_exp_last||: 0.0673


Computing sample gradients:   1%|▏         | 2/157 [01:05<1:25:12, 32.99s/it]

In [ ]:
# When a parent process starts a child process via subprocess, the two are separate entities with their own memory space.
# The parent process is not notified in real-time about the filesystem modifications the child process is making.